<a href="https://colab.research.google.com/github/alj-x64/Realtime-Bass-Tablature-Transcription/blob/main/Colab_Master_Training_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.optim as optim
import time
import numpy as np
import optuna
import csv
import os

# Import your custom mathematical engine and model
from adaptive_hpo import AdaptiveMultiStageHPO
from tabcnn_model import DynamicTabCNN

# ==========================================
# 1. THE SHARED PYTORCH TRAINING PIPELINE
# ==========================================
def evaluate_model_pipeline(config, stress_test=False):
    """
    Ito ang NAG-IISANG training loop para sa thesis mo.
    Gagamitin ito pareho ng Proposed Algorithm mo at ng Optuna (Random/Bayesian).
    Garantisadong fair ang comparison!
    """
    model = DynamicTabCNN(config)
    # Ilipat agad sa GPU
    if torch.cuda.is_available():
        model = model.to('cuda')
        
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
    
    # --- SIMULATED TRAINING LOOP ---
    # (Ilalagay mo rito ang totoong PyTorch dataloader for loop mo)
    # ...
    
    # --- EVALUATION AND LATENCY CALCULATION (Eq 20) ---
    model.eval()
    dummy_input = torch.randn(1, 4608) # Raw audio 209ms context window (18 frames * 256 hop)
    if torch.cuda.is_available():
        dummy_input = dummy_input.to('cuda')
        
    latencies = []
    
    with torch.no_grad():
        for _ in range(10): # Simulate evaluating 10 frames
            # --- GAUSSIAN NOISE INJECTION (STRESS TEST) ---
            if stress_test:
                noise_std = 0.05 
                gaussian_noise = torch.randn_like(dummy_input) * noise_std
                network_input = dummy_input + gaussian_noise
            else:
                network_input = dummy_input

            t_i = time.perf_counter()
            _ = model(network_input) 
            t_o = time.perf_counter()
            
            delta_t_ms = (t_o - t_i) * 1000
            latencies.append(delta_t_ms)
            
    avg_latency = sum(latencies) / len(latencies)
    
    validation_loss = np.random.uniform(0.1, 0.4) 
            
    return validation_loss, avg_latency

# ==========================================
# 2. OPTUNA OBJECTIVE & LOGGER WRAPPERS
# ==========================================
def optuna_objective(trial):
    """Ito ang tulay para maintindihan ni Optuna yung config format natin."""
    config = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
        'activation': trial.suggest_categorical('activation', ['ReLU', 'Tanh', 'ELU']),
        'conv_layers': trial.suggest_int('conv_layers', 1, 4),
        'filter_layers': trial.suggest_categorical('filter_layers', [16, 32, 64, 128]),
        'kernel_size': trial.suggest_categorical('kernel_size', [2, 3, 5, 7])
    }
    
    loss, latency = evaluate_model_pipeline(config, stress_test=False)
    
    latency_penalty = (latency / 200.0) if latency > 0 else 0 
    fitness = 0.5 * (1.0 - loss) - 0.5 * latency_penalty
    
    return fitness

def optuna_csv_logger(study, trial, filename):
    """
    Custom callback para i-log sa CSV ang bawat galaw ni Optuna.
    Garantisadong pantay ang tracking mo sa custom HPO at sa baselines.
    """
    file_exists = os.path.isfile(filename)
    with open(filename, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            # CSV Headers
            writer.writerow([
                'Trial', 'LearningRate', 'DropoutRate', 'Activation', 
                'ConvLayers', 'FilterLayers', 'KernelSize', 'Fitness'
            ])
        
        # Isulat ang resulta ng katatapos lang na trial
        writer.writerow([
            trial.number + 1,
            f"{trial.params['learning_rate']:.6f}",
            trial.params['dropout_rate'],
            trial.params['activation'],
            trial.params['conv_layers'],
            trial.params['filter_layers'],
            trial.params['kernel_size'],
            f"{trial.value:.4f}"
        ])

# ==========================================
# 3. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    print("--- Thesis Optimization Experiments ---")
    
    # Choices: "PROPOSED", "RANDOM", "BAYESIAN"
    MODE = "BAYESIAN" 
    
    TOTAL_TRIALS = 30 # Standardized budget
    
    if MODE == "PROPOSED":
        print("\n>>> RUNNING PROPOSED ADAPTIVE HPO <<<")
        hpo_engine = AdaptiveMultiStageHPO(total_budget=TOTAL_TRIALS, log_file="hpo_proposed_log.csv")
        best_config = hpo_engine.run_optimization(evaluate_model_pipeline)
        print("\nBest Config from Proposed Algo:", best_config)
        save_name = "trained_prop.pth"

    elif MODE == "RANDOM":
        print("\n>>> RUNNING OPTUNA RANDOM SEARCH <<<")
        log_filename = "optuna_random_log.csv"
        # Linisin ang lumang log kung meron
        if os.path.exists(log_filename): os.remove(log_filename)
            
        sampler = optuna.samplers.RandomSampler()
        study = optuna.create_study(direction="maximize", sampler=sampler)
        # Naka-inject ang custom CSV logger callback dito!
        study.optimize(optuna_objective, n_trials=TOTAL_TRIALS, 
                       callbacks=[lambda s, t: optuna_csv_logger(s, t, log_filename)])
        
        best_config = study.best_params
        print("\nBest Config from Random Search:", best_config)
        save_name = "trained_random.pth"

    elif MODE == "BAYESIAN":
        print("\n>>> RUNNING OPTUNA BAYESIAN OPTIMIZATION (TPE) <<<")
        log_filename = "optuna_bayesian_log.csv"
        # Linisin ang lumang log kung meron
        if os.path.exists(log_filename): os.remove(log_filename)
            
        sampler = optuna.samplers.TPESampler()
        study = optuna.create_study(direction="maximize", sampler=sampler)
        # Naka-inject ang custom CSV logger callback dito!
        study.optimize(optuna_objective, n_trials=TOTAL_TRIALS, 
                       callbacks=[lambda s, t: optuna_csv_logger(s, t, log_filename)])
        
        best_config = study.best_params
        print("\nBest Config from Bayesian Opt:", best_config)
        save_name = "trained_bayesian.pth"

    # ==========================================
    # 4. FINAL RETRAINING AND SAVING
    # ==========================================
    print(f"\n>>> RETRAINING FINAL MODEL USING BEST {MODE} CONFIG <<<")
    final_model = DynamicTabCNN(best_config)
    if torch.cuda.is_available():
        final_model = final_model.to('cuda')
        
    torch.save(final_model.state_dict(), save_name)
    
    import json
    with open(save_name.replace('.pth', '_config.json'), 'w') as f:
        json.dump(best_config, f, indent=4)
        
    print(f"✅ Final model weights saved to: {save_name}")
    print(f"✅ Final model config saved to: {save_name.replace('.pth', '_config.json')}")